> **Course website copy · Team 1.** This is the team's final submission as handed in, with their run outputs. Names are removed. The one cell below was added for the website: it fetches the team's data files from the course repository, so *Runtime → Run all* works without uploading anything. Everything after it is the team's own work.

In [ ]:
# Added for the course website (not part of the team's submission):
# copy the team's data files from the course repository into the folder the notebook expects.
import os, shutil, subprocess
SRC = "week07/projects/team1-office-chair/data"
TARGET = "my_product_data"
if not os.path.isdir("course_copy"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/TH-Koln-Bartnik/sscm-2026-student", "course_copy"], check=True)
    subprocess.run(["git", "-C", "course_copy", "sparse-checkout", "set", SRC], check=True)
os.makedirs(TARGET, exist_ok=True)
for name in os.listdir(os.path.join("course_copy", SRC)):
    shutil.copy(os.path.join("course_copy", SRC, name), os.path.join(TARGET, name))
print(len(os.listdir(TARGET)), "data files in", os.path.abspath(TARGET))


# SSCM 2026 · Group project · Team 1 · Office chair (Interstuhl EVERY POLSTER EV15R)

One year (52 weeks) of the chair line, fed with gas springs by its key supplier. Three scenarios, each with the 10 seeds 1–10:

1. **Baseline:** gas springs from Ningbo (CN) by sea via the Suez Canal and by truck, normal lead times.
2. **Disruption:** the Suez route is unusable for 12 weeks (Red Sea diversions 2024). Ships that would pass Suez in this window sail around the Cape of Good Hope instead (+10–14 days, +3,400 nautical miles).
3. **Disruption + countermeasure:** the plant switches to Stabilus (Koblenz, DE), delivery by truck.

**How to run:** Runtime → Run all. Cell 1 asks for the 13 CSV files if they are not in the folder yet. The notebook writes `results.json` and `animation.html`. The official check cell sits under "Step 4".

In [1]:
# Step 1 · Data: put the CSV files into the folder my_product_data
import os
DATA_DIR = "my_product_data"
FILES = ["products", "bom", "workcenters", "routing", "process_environment", "suppliers", "locations",
         "network_lanes", "freight_rates", "sources", "emission_factors", "transport_emission_factors",
         "transport_modes"]
os.makedirs(DATA_DIR, exist_ok=True)
missing = [f + ".csv" for f in FILES if not os.path.exists(os.path.join(DATA_DIR, f + ".csv"))]
if missing:
    from google.colab import files          # dialog: select all 13 CSV files at once
    for name, content in files.upload().items():
        open(os.path.join(DATA_DIR, name), "wb").write(content)

# sources.csv: remove spaces after the commas in the header line (" row_id" -> "row_id"); harmless if already clean
p = os.path.join(DATA_DIR, "sources.csv")
lines = open(p, encoding="utf-8").read().split("\n")
lines[0] = ",".join(c.strip() for c in lines[0].split(","))
open(p, "w", encoding="utf-8").write("\n".join(lines))
print(sorted(os.listdir(DATA_DIR)))

Saving bom.csv to bom.csv
Saving emission_factors.csv to emission_factors.csv
Saving freight_rates.csv to freight_rates.csv
Saving locations.csv to locations.csv
Saving network_lanes.csv to network_lanes.csv
Saving process_environment.csv to process_environment.csv
Saving products.csv to products.csv
Saving routing.csv to routing.csv
Saving sources.csv to sources.csv
Saving suppliers.csv to suppliers.csv
Saving transport_emission_factors.csv to transport_emission_factors.csv
Saving transport_modes.csv to transport_modes.csv
Saving workcenters.csv to workcenters.csv
['bom.csv', 'emission_factors.csv', 'freight_rates.csv', 'locations.csv', 'network_lanes.csv', 'process_environment.csv', 'products.csv', 'routing.csv', 'sources.csv', 'suppliers.csv', 'transport_emission_factors.csv', 'transport_modes.csv', 'workcenters.csv']


## Step 2 · Model choices that are not in the CSV files

Everything about stations, bill of materials, suppliers, lanes, freight rates and emission factors is read from the CSV files. The table lists the choices made in the model itself.

| Parameter | Value | Reason |
|---|---|---|
| Demand | 700 chairs per week | `sources.csv`: about 3,000 chairs per month, 3,000 × 12 / 52 |
| Clock | working hours (40 h per week); supplier lead times in working days of 8 h (the lane times in `network_lanes.csv` are calendar days: Ningbo chain 48 calendar days × 5/7 = 34.3, set to 35 working days) | one clock for everything, as the Little's Law test requires |
| Availability | stretches the process time (time / availability) | consistent with the capacity formula of test T2 |
| Order rule | order up to: stock + on order = demand × (lead time + review period + safety stock) | standard periodic-review rule |
| Review period | CN every 2 weeks, DE every week | sea freight is shipped in larger lots than domestic trucks |
| Safety stock | 1.0 week of demand | lead-time standard deviation of the sea route is 0.7 weeks (3.5 working days) |
| WIP limit | 60 chairs on the line | stops the line from piling up material in front of the station that waits for parts |
| Waiting demand | at most 700 chairs (1 week of demand) wait; new demand beyond that is lost | otherwise the line makes up every stop before year end |
| Hours stopped | hours in which the key station (WC3) has a chair waiting for a gas spring, the stock is empty and no server of WC3 is working | the line cannot finish chairs; hours with some demand waiting are not counted |
| Suez passage | at 65 % of the total lead time | share of the Shanghai–Rotterdam route that lies before the canal |
| Countermeasure | from week 16 all orders go to Stabilus, until the end of the year | one week of reaction time after the diversions start |

In [2]:
# Parameters (change them here, nothing else in the notebook needs to be touched)
P = dict(
    weeks=52, seeds=list(range(1, 11)),
    hold_rate=0.20,            # course value: 20 % inventory holding rate per year
    co2_price=100.0,           # course value: 100 EUR per tonne CO2e
    demand_per_week=700,       # chairs per week
    conwip=60,                 # maximum chairs on the line at the same time
    safety_weeks=1.0,          # safety stock of gas springs in weeks of demand
    max_backlog=700,           # demand that may wait (the rest is lost)
    review_weeks={"SUP-CN": 2, "SUP-DE": 1},   # review period per supplier in weeks
    suez_start_week=14,        # the Suez route is unusable from the start of this week (0 = first week)
    suez_closed_weeks=12,      # length of the closure in weeks
    suez_extra_days=(10, 14),  # extra calendar days of the Cape route (reported range)
    suez_extra_km=3400 * 1.852,# extra distance of the Cape route: 3,400 nautical miles in km
    suez_share=0.65,           # share of the lead time at which a ship reaches the Suez Canal
    scenarios={
        "baseline":       dict(label="Baseline",                          suez=False, switch_week=None),
        "disruption":     dict(label="Suez closed (Red Sea diversion)",   suez=True,  switch_week=None),
        "countermeasure": dict(label="Suez closed + German supplier",     suez=True,  switch_week=15),
    })

**Real event behind the disruption (Red Sea diversions, 2024).** Houthi attacks on merchant ships in the Red Sea began in November 2023. Maersk and Hapag-Lloyd stopped their Red Sea passages on 15 December 2023, and Maersk paused all Red Sea transits until further notice on 2 January 2024. The ships sailed around the Cape of Good Hope instead, which adds about 10–14 days (Drewry; Wallenius Wilhelmsen) and about 3,400 nautical miles (C.H. Robinson). The effect on a German plant: Tesla stopped most car production at Grünheide from 29 January to 11 February 2024 for lack of components that arrived late from Asia.

The diversions lasted much longer than our window. The model uses **12 weeks**, which is our own assumption (weeks 15 to 26 of the plant's year). The Ever Given blockage of 2021 (six days) was considered and rejected, because the sea route stayed usable afterwards and a one-week effect disappears in the safety stock. Sources are listed at the end of the notebook.

In [3]:
import os, math, json, heapq, collections
import numpy as np
import pandas as pd

# ------------------------------------------------------------------ 1. DATA
def load_data(DATA_DIR):
    rd = lambda f: pd.read_csv(os.path.join(DATA_DIR, f))
    D = {n: rd(n + ".csv") for n in ["products", "bom", "workcenters", "routing", "process_environment",
                                      "suppliers", "locations", "network_lanes", "freight_rates",
                                      "emission_factors", "transport_emission_factors", "transport_modes"]}
    return D

def build_model(D, P):
    """Turns the CSV files into plain Python numbers. Nothing is typed in by hand here."""
    prod = D["products"].iloc[0];  pid = prod["product_id"]
    bom, sup = D["bom"], D["suppliers"]
    key = bom[(bom.product_id == pid) & bom.component_id.isin(sup.component_id)]
    assert len(key) == 1, "expected exactly one key component"
    key = key.iloc[0]
    M = dict(product_id=pid, product=prod["product_name"], price=float(prod["list_price_eur"]),
             key_comp=key["component_id"], key_name=key["component_name"], key_qty=float(key["qty_per_unit"]),
             key_mass=float(key["mass_kg"]), key_wc=key["added_at_workcenter"],
             base_supplier=key["supplier_id"])
    other = bom[(bom.product_id == pid) & (bom.component_id != key["component_id"])]
    M["other_material"] = float((other.qty_per_unit * other.unit_cost_eur).sum())

    # stations in routing order
    wc = D["workcenters"].set_index("workcenter_id")
    rt = D["routing"][D["routing"].product_id == pid].sort_values("seq")
    env = D["process_environment"].set_index("workcenter_id")
    st = []
    for r in rt.itertuples():
        w = wc.loc[r.workcenter_id]
        st.append(dict(id=r.workcenter_id, name=w["workcenter_name"], c=int(w["num_servers"]),
                       cv=float(w["process_time_cv"]),
                       mean_h=float(r.process_time_min) / 60.0 / float(w["availability"]),  # availability stretches the time
                       fixed_week=float(w["fixed_cost_eur_week"]),
                       kw_run=float(env.loc[r.workcenter_id, "kw_running"]),
                       kw_idle=float(env.loc[r.workcenter_id, "kw_idle"]),
                       shift_h=float(w["shift_hours"]), days=float(w["days_per_week"])))
    M["st"] = st
    M["key_idx"] = [s["id"] for s in st].index(M["key_wc"])
    M["H_WEEK"] = st[0]["shift_h"] * st[0]["days"]           # working hours per week (clock of the whole model)
    M["H_DAY"] = st[0]["shift_h"]
    assert all(s["shift_h"] * s["days"] == M["H_WEEK"] for s in st), "stations with different calendars"

    # emission factors
    ef = D["emission_factors"].set_index("factor_id")["value"]
    loc = D["locations"].set_index("location_id")
    plant = loc[loc.type == "plant"].index[0]
    M["plant"] = plant
    M["ef_el"] = float(ef[[i for i in ef.index if i.startswith("grid_electricity_" + loc.loc[plant, "country"])][0]])
    tef = D["transport_emission_factors"].set_index("mode_id")["g_co2e_per_tkm"].astype(float)
    lanes = D["network_lanes"].set_index("origin_id")
    rates = D["freight_rates"].set_index("lane_id")

    # suppliers: lane chain supplier -> plant, freight and transport CO2 per shipment
    S = {}
    for r in sup[sup.component_id == M["key_comp"]].itertuples():
        chain, cur = [], r.location_id
        while cur != plant:
            ln = lanes.loc[cur]; chain.append(ln); cur = ln["destination_id"]
        S[r.supplier_id] = dict(name=r.supplier_name, country=r.country, price=float(r.unit_price_eur),
                                mean_wd=float(r.lead_time_mean_days), sd_wd=float(r.lead_time_sd_days),
                                co2_embodied=float(r.kg_co2e_per_unit), chain=chain,
                                lane_ids=[l["lane_id"] for l in chain],
                                via_sea=any(l["mode_id"] == "MODE-SEA" for l in chain))
    M["S"] = S
    M["alt_supplier"] = [k for k in S if k != M["base_supplier"]][0]

    def shipment(sid, qty, extra_km=0.0):
        """Freight cost (EUR) and transport CO2 (kg) of one shipment; extra_km is added to the sea lane (Cape detour)."""
        t = qty * M["key_mass"] / 1000.0
        eur = co2 = 0.0
        for l in S[sid]["chain"]:
            fr = rates.loc[l["lane_id"]]
            km = float(l["distance_km"]) + (extra_km if l["mode_id"] == "MODE-SEA" else 0.0)
            eur += max(float(fr["min_charge_eur"]), float(fr["rate_eur_per_shipment"]) + float(fr["rate_eur_per_tkm"]) * t * km)
            co2 += t * km * tef[l["mode_id"]] / 1000.0
        return eur, co2
    M["shipment"] = shipment
    return M

# ------------------------------------------------------------------ 2. SIMULATION
def simulate(M, P, scenario, seed):
    cfg = P["scenarios"][scenario]
    st, n, K = M["st"], len(M["st"]), M["key_idx"]
    H, HD, WEEKS = M["H_WEEK"], M["H_DAY"], P["weeks"]
    END = WEEKS * H
    D_WK = P["demand_per_week"]
    rng_p = [np.random.default_rng([seed, 100 + i]) for i in range(n)]   # one stream per station -> same noise in all scenarios
    rng_l = np.random.default_rng([seed, 7]);  rng_c = np.random.default_rng([seed, 8])
    sig = [math.sqrt(math.log(1 + s["cv"] ** 2)) for s in st]
    mu = [math.log(s["mean_h"]) - sig[i] ** 2 / 2 for i, s in enumerate(st)]
    S, SH = M["S"], M["shipment"]
    B0, B1 = P["suez_start_week"] * H, (P["suez_start_week"] + P["suez_closed_weeks"]) * H

    ev, seq = [], 0
    def push(t, kind, a=None, b=None):
        nonlocal seq; seq += 1; heapq.heappush(ev, (t, seq, kind, a, b))

    queue = [collections.deque() for _ in range(n)]; busy = [0] * n
    lots = collections.deque()                       # FIFO stock: [supplier, qty, price, freight/unit, co2_emb/unit, co2_tr/unit]
    stock = 0.0; stock_val = 0.0; transit = 0.0; transit_val = 0.0
    pipe = {k: 0.0 for k in S}
    backlog = 0; wip = 0; started = 0; finished = 0; used = 0
    start_h, finish_h = [], []
    acc = dict(mat=0.0, fr=0.0, emb=0.0, tr=0.0, hold=0.0)
    q_int = [0.0] * n; busy_h = [0.0] * n; stop_h = 0.0
    wq = [0.0] * n; w_stop = 0.0; w_fin = 0
    W = dict(queue=[], in_transit=[], stock=[], finished=[], stopped_hours=[], supplier_down=[], supplier=[])
    last = 0.0

    def advance(t):
        nonlocal last, stop_h, w_stop
        dt = t - last
        if dt <= 0: return
        for i in range(n):
            q_int[i] += len(queue[i]) * dt; busy_h[i] += busy[i] * dt; wq[i] += len(queue[i]) * dt
        if queue[K] and stock <= 0 and busy[K] == 0:
            stop_h += dt; w_stop += dt
        acc["hold"] += (stock_val + transit_val) * dt
        last = t

    def try_start(i, t):
        nonlocal stock, stock_val, used
        while busy[i] < st[i]["c"] and queue[i]:
            if i == K:
                if stock <= 0: break
                lot = lots[0]; lot[1] -= 1; stock -= 1; stock_val -= lot[2]; used += 1
                acc["mat"] += lot[2]; acc["fr"] += lot[3]; acc["emb"] += lot[4]; acc["tr"] += lot[5]
                if lot[1] <= 1e-9: lots.popleft()
            u = queue[i].popleft(); busy[i] += 1
            push(t + rng_p[i].lognormal(mu[i], sig[i]), 1, i, u)

    def release(t):
        nonlocal backlog, wip, started
        while backlog > 0 and wip < P["conwip"]:
            backlog -= 1; wip += 1; started += 1
            start_h.append(t); finish_h.append(math.inf)
            queue[0].append(len(start_h) - 1)
        try_start(0, t)

    def new_lot(sid, qty, extra_km=0.0):
        eur, co2 = SH(sid, qty, extra_km)
        return [sid, float(qty), S[sid]["price"], eur / qty, S[sid]["co2_embodied"], co2 / qty]

    def active_supplier(w):
        sw = cfg["switch_week"]
        return M["alt_supplier"] if (sw is not None and w >= sw) else M["base_supplier"]

    # ---- start state: a plant that has been running normally (steady-state stock and pipeline)
    b = M["base_supplier"]; Lw = S[b]["mean_wd"] / 5.0; R = P["review_weeks"][b]
    j, pq = 1, 0.0
    while Lw - j * R > 1e-9:
        q = D_WK * R; lot = new_lot(b, q); pipe[b] += q; transit += q; transit_val += q * lot[2]
        push((Lw - j * R) * H, 3, lot); j += 1; pq += q
    q0 = max(0.0, D_WK * (Lw + P["safety_weeks"]) - pq)
    if q0 > 0:
        lots.append(new_lot(b, q0)); stock = q0; stock_val = q0 * lots[0][2]
    push(0.0, 2, 0); push(H / D_WK, 0); push(H, 4, 0)
    next_demand = 1

    while ev:
        t, _, kind, a, bb = heapq.heappop(ev)
        if t > END + 1e-9: break
        advance(t)
        if kind == 0:                                     # demand: one more chair ordered
            backlog = min(backlog + 1, P["max_backlog"]); release(t)
            next_demand += 1; push(next_demand * H / D_WK, 0)
        elif kind == 1:                                   # a station finished one unit
            i, u = a, bb; busy[i] -= 1
            if i < n - 1:
                queue[i + 1].append(u); try_start(i + 1, t)
            else:
                finish_h[u] = t; finished += 1; w_fin += 1; wip -= 1
            try_start(i, t); release(t)
        elif kind == 2:                                   # weekly review of the key-component stock
            w = a
            if w < WEEKS:
                sid = active_supplier(w)
                if w % P["review_weeks"][sid] == 0:
                    s = S[sid]; Lw = s["mean_wd"] / 5.0
                    target = D_WK * (Lw + P["review_weeks"][sid] + P["safety_weeks"])
                    qty = math.ceil(target - (stock + pipe[sid]))
                    if qty > 0:
                        L = max(1.0, rng_l.normal(s["mean_wd"], s["sd_wd"])) * HD   # lead time in working hours
                        passage = t + P["suez_share"] * L
                        detour = 0.0
                        if cfg["suez"] and s["via_sea"] and B0 <= passage < B1:
                            # Suez not usable: the ship sails around the Cape (extra days drawn from the reported 10-14 d)
                            extra_wd = rng_c.uniform(*P["suez_extra_days"]) * 5.0 / 7.0
                            arr = t + L + extra_wd * HD; detour = P["suez_extra_km"]
                        else:
                            arr = t + L
                        lot = new_lot(sid, qty, detour); pipe[sid] += qty; transit += qty; transit_val += qty * lot[2]
                        push(arr, 3, lot)
                push((w + 1) * H, 2, w + 1)
        elif kind == 3:                                   # a shipment of key components arrives at the plant
            lot = a; q = lot[1]
            pipe[lot[0]] -= q; transit -= q; transit_val -= q * lot[2]
            lots.append(lot); stock += q; stock_val += q * lot[2]
            try_start(K, t)
        elif kind == 4:                                   # end of week: snapshot for the animation
            w = a
            W["queue"].append([round(x / H, 2) for x in wq]); wq = [0.0] * n
            W["in_transit"].append(round(transit)); W["stock"].append(round(stock))
            W["finished"].append(w_fin); w_fin = 0
            W["stopped_hours"].append(round(w_stop, 2)); w_stop = 0.0
            W["supplier_down"].append(bool(cfg["suez"] and w * H < B1 and (w + 1) * H > B0))
            W["supplier"].append(active_supplier(w))
            if w + 1 < WEEKS: push((w + 2) * H, 4, w + 1)
    advance(END)

    # ---- the books of the year
    fixed = sum(s["fixed_week"] for s in st) * WEEKS
    other_mat = started * M["other_material"]
    kwh = sum(s["kw_run"] * busy_h[i] + s["kw_idle"] * (s["c"] * END - busy_h[i]) for i, s in enumerate(st))
    co2_el = kwh * M["ef_el"]
    co2_kg = acc["emb"] + acc["tr"] + co2_el
    hold = P["hold_rate"] * acc["hold"] / END * (WEEKS / 52.0)
    co2_cost = co2_kg / 1000.0 * P["co2_price"]
    total = fixed + other_mat + acc["mat"] + acc["fr"] + hold + co2_cost
    s_arr, f_arr = np.asarray(start_h, float), np.asarray(finish_h, float)
    done = f_arr <= END
    return dict(
        weeks_simulated=WEEKS, hours_simulated=END, units_started=started, units_finished=finished,
        key_components_used=used,
        avg_units_on_line=round(float((np.minimum(f_arr, END) - s_arr).sum() / END), 2),
        avg_hours_on_line=round(float((f_arr[done] - s_arr[done]).mean()), 2),
        cost_per_unit_eur=round(total / finished, 2),
        # scoreboard
        units_per_week=finished / WEEKS, kg_co2e_per_unit=co2_kg / finished, hours_stopped=stop_h,
        # breakdown (per unit finished)
        parts=dict(fixed=fixed / finished, other_material=other_mat / finished, key_material=acc["mat"] / finished,
                   freight=acc["fr"] / finished, holding=hold / finished, co2_cost=co2_cost / finished),
        co2_parts=dict(key_component=acc["emb"] / finished, transport=acc["tr"] / finished, electricity=co2_el / finished),
        weekly=W)


# Build the model from the CSV files and print what was read
D = load_data(DATA_DIR)
M = build_model(D, P)
print("Product:", M["product"], "· list price", M["price"], "EUR")
print("Key component:", M["key_name"], "· added at", M["key_wc"], "· baseline supplier:", M["base_supplier"],
      "· alternative:", M["alt_supplier"])
for sid, s in M["S"].items():
    q = P["demand_per_week"] * P["review_weeks"][sid]
    eur, co2 = M["shipment"](sid, q)
    print(f"{sid}: {s['name']} · lanes {s['lane_ids']} · lead time {s['mean_wd']} ± {s['sd_wd']} working days · "
          f"freight {eur / q:.2f} EUR/unit · transport {co2 / q:.3f} kg CO2e/unit")

Product: Interstuhl EVERY POLSTER Aktionsmodell EV15R · list price 310.08 EUR
Key component: Gas Spring Class 4 · added at WC3 · baseline supplier: SUP-CN · alternative: SUP-DE
SUP-CN: Ningbo Anatec Gas Spring Co., Ltd. · lanes ['LANE-CN-SHA', 'LANE-SHA-RTM', 'LANE-RTM-PLANT'] · lead time 35.0 ± 3.5 working days · freight 7.45 EUR/unit · transport 0.464 kg CO2e/unit
SUP-DE: Stabilus GmbH Koblenz · lanes ['LANE-DE-PLANT'] · lead time 4.5 ± 1.0 working days · freight 0.25 EUR/unit · transport 0.024 kg CO2e/unit


**Which station limits the line?** Time per chair per place = process time ÷ (places × availability). The station with the largest value is the bottleneck.

In [4]:
bn = pd.DataFrame([dict(station=s["id"], name=s["name"], places=s["c"],
                        min_per_chair_per_place=s["mean_h"] * 60 / s["c"],
                        capacity_per_week=s["c"] * M["H_WEEK"] / s["mean_h"],
                        load_at_demand=P["demand_per_week"] * s["mean_h"] / (s["c"] * M["H_WEEK"]))
                  for s in M["st"]]).set_index("station")
bn["load_at_demand"] = (bn["load_at_demand"] * 100).round(1).astype(str) + " %"
display(bn.round(2))
print("Bottleneck:", bn["min_per_chair_per_place"].idxmax())

,name,places,min_per_chair_per_place,capacity_per_week,load_at_demand
station,,,,,
WC1,PU Foam Moulding & Trimming,2,2.78,864.00,81.0 %
WC2,Upholstery & Fabric Cover Assembly,2,2.50,960.00,72.9 %
WC3,Gas Spring Base Casters & Tilt Mechanism Sub-A...,3,2.67,900.00,77.8 %
WC4,Backrest Joining Armrest & Ergonomic Test,2,2.28,1051.43,66.6 %
WC5,Packaging,1,2.33,1028.57,68.1 %


Bottleneck: WC1


## Step 3 · Three scenarios × 10 seeds

In [5]:
import time
t0 = time.time()
RESULTS = {sc: [simulate(M, P, sc, seed) for seed in P["seeds"]] for sc in P["scenarios"]}
print(f"done in {time.time() - t0:.0f} s: {sum(len(v) for v in RESULTS.values())} runs")

done in 39 s: 30 runs


In [6]:
# Scoreboard: mean (lowest – highest value over the 10 seeds)
SCORE = [("units_per_week", "chairs per week", 1), ("cost_per_unit_eur", "total cost per unit [EUR]", 2),
         ("kg_co2e_per_unit", "kg CO2e per unit", 2), ("hours_stopped", "hours line stood still", 1)]
def stat(sc, key):
    v = np.array([r[key] for r in RESULTS[sc]], float); return v.mean(), v.min(), v.max()
table = pd.DataFrame({P["scenarios"][sc]["label"]: {name: "%.*f  (%.*f – %.*f)" % (d, stat(sc, k)[0], d, stat(sc, k)[1], d, stat(sc, k)[2])
                                                    for k, name, d in SCORE} for sc in P["scenarios"]}).T
display(table)

# What the cost per unit is made of (mean over the seeds, EUR per finished chair)
parts = pd.DataFrame({P["scenarios"][sc]["label"]: {k: np.mean([r["parts"][k] for r in RESULTS[sc]]) for k in RESULTS[sc][0]["parts"]}
                      for sc in P["scenarios"]}).T
parts["total"] = parts.sum(axis=1)
display(parts.round(2))
# What the CO2 per unit is made of (kg CO2e per finished chair)
co2 = pd.DataFrame({P["scenarios"][sc]["label"]: {k: np.mean([r["co2_parts"][k] for r in RESULTS[sc]]) for k in RESULTS[sc][0]["co2_parts"]}
                    for sc in P["scenarios"]}).T
co2["total"] = co2.sum(axis=1)
display(co2.round(3))

,chairs per week,total cost per unit [EUR],kg CO2e per unit,hours line stood still
Baseline,699.8 (699.8 – 699.9),149.93 (149.93 – 149.94),4.60 (4.60 – 4.60),26.8 (0.0 – 53.7)
Suez closed (Red Sea diversion),690.6 (679.0 – 699.8),150.49 (150.24 – 150.89),4.63 (4.62 – 4.64),133.1 (98.6 – 217.6)
Suez closed + German supplier,699.8 (699.8 – 699.9),152.55 (152.54 – 152.55),4.19 (4.18 – 4.19),8.3 (0.0 – 42.9)


,fixed,other_material,key_material,freight,holding,co2_cost,total
Baseline,16.55,115.13,10.00,7.45,0.35,0.46,149.93
Suez closed (Red Sea diversion),16.77,115.14,10.00,7.76,0.35,0.46,150.49
Suez closed + German supplier,16.55,115.13,16.46,3.73,0.26,0.42,152.55


,key_component,transport,electricity,total
Baseline,3.260,0.464,0.873,4.597
Suez closed (Red Sea diversion),3.260,0.490,0.877,4.628
Suez closed + German supplier,3.072,0.240,0.873,4.185


**Total cost** (one place in the code: the end of `simulate`) = station fixed costs (`fixed_cost_eur_week` × 52) + the other bill-of-material parts for all chairs started + gas springs (price of the supplier the spring came from) + freight (lanes of that supplier; per lane and shipment `max(min_charge, fixed rate + rate per tkm × tonnes × km)`) + **0.20 × average value of gas springs in stock and on the way** + **100 EUR per tonne CO2e**. Cost per unit = total cost ÷ chairs finished. Price, freight and CO2 of a gas spring are booked when the spring is used in the line (first in, first out). Electricity is not charged in EUR (no price in the data); its CO2 is.

**CO2 per unit** = gas spring (`kg_co2e_per_unit` of the supplier) + transport of the gas spring (tonne-kilometres × g CO2e per tkm of each mode, read from `transport_emission_factors.csv`) + factory electricity (`kw_running` and `kw_idle` of every place from `process_environment.csv` × hours × grid factor from `emission_factors.csv`).

In [7]:
# Sensitivity for the poster: what happens to chairs per week if the time per chair at WC1 rises by 20 %?
import copy
M2 = copy.deepcopy(M); M2["shipment"] = M["shipment"]
M2["st"][0]["mean_h"] *= 1.2
runs20 = [simulate(M2, P, "baseline", seed) for seed in P["seeds"]]
print("WC1 +20 %: capacity", round(M2["st"][0]["c"] * M2["H_WEEK"] / M2["st"][0]["mean_h"]), "chairs/week; chairs per week:",
      round(np.mean([r["units_per_week"] for r in runs20]), 1), "(baseline", round(stat("baseline", "units_per_week")[0], 1), ")")

WC1 +20 %: capacity 720 chairs/week; chairs per week: 698.2 (baseline 699.8 )


## Step 3b · Three questions from the checkpoint review

The same model, the same data and the same 10 seeds. Each line below changes exactly one thing. Part 1 asks why the plant buys in Ningbo at all, part 2 where the stand-still hours of the baseline come from, part 3 what warns the plant early in the disruption.

In [8]:
# Step 3b · Questions from the checkpoint review: extra runs with the same model, the same data and the same 10 seeds
def variant(P2, M2, cfg):
    """Mean (lowest - highest) over the 10 seeds of one extra scenario cfg."""
    P3 = copy.deepcopy(P2); P3["scenarios"]["extra"] = cfg
    rs = [simulate(M2, P3, "extra", seed) for seed in P3["seeds"]]
    return {k: np.array([r[k] for r in rs], float) for k in ("units_per_week", "cost_per_unit_eur", "kg_co2e_per_unit", "hours_stopped")}
def line(label, v):
    print(f"{label:<47}{v['units_per_week'].mean():7.1f} chairs/week {v['cost_per_unit_eur'].mean():8.2f} EUR {v['kg_co2e_per_unit'].mean():6.2f} kg CO2e"
          f"{v['hours_stopped'].mean():7.1f} h stopped ({v['hours_stopped'].min():.1f} - {v['hours_stopped'].max():.1f})")
as_arrays = lambda sc: {k: np.array([r[k] for r in RESULTS[sc]], float) for k in ("units_per_week", "cost_per_unit_eur", "kg_co2e_per_unit", "hours_stopped")}
NO_DISRUPTION = dict(label="extra", suez=False)

print("(1) Why does the plant buy in Ningbo at all?  Same year without any disruption, German supplier from week 1")
line("Baseline: Ningbo, no disruption", as_arrays("baseline"))
line("German supplier all year, no disruption", variant(P, M, dict(NO_DISRUPTION, switch_week=0)))

# Landed cost of one gas spring = price + freight per spring (freight recomputed here from freight_rates.csv and checked against the model)
fr = pd.read_csv(f"{DATA_DIR}/freight_rates.csv").set_index("lane_id")
def landed(sid, sea_per_tkm=None):
    q = P["demand_per_week"] * P["review_weeks"][sid] * M["key_qty"]            # one regular shipment (units)
    t, eur = q * M["key_mass"] / 1000.0, 0.0
    for l in M["S"][sid]["chain"]:
        f = fr.loc[l["lane_id"]]
        r = f["rate_eur_per_tkm"] if not (l["mode_id"] == "MODE-SEA" and sea_per_tkm is not None) else sea_per_tkm
        eur += max(float(f["min_charge_eur"]), float(f["rate_eur_per_shipment"]) + float(r) * t * float(l["distance_km"]))
    if sea_per_tkm is None: assert abs(eur - M["shipment"](sid, q)[0]) < 1e-6   # same number as inside the simulation
    return M["S"][sid]["price"], eur / q
for sid in M["S"]:
    pr, fq = landed(sid); print(f"   {M['S'][sid]['name']:<22} price {pr:5.2f} + freight {fq:5.2f} = {pr + fq:5.2f} EUR per spring delivered")
pr, fq = landed("SUP-CN", sea_per_tkm=0.0)
print(f"   Ningbo, sea lane at the container rate only (no 0.18 EUR/tkm part): {pr:5.2f} + {fq:5.2f} = {pr + fq:5.2f} EUR per spring delivered")
print("   Which Stabilus price makes the switch (week 16, Suez closed) cost the same per chair as the baseline?")
prices, costs = [14.5, 17.0, 17.5, 22.0], []
for price in prices:
    M2 = copy.deepcopy(M); M2["S"]["SUP-DE"]["price"] = price
    costs.append(variant(P, M2, P["scenarios"]["countermeasure"])["cost_per_unit_eur"].mean())
    print(f"      Stabilus {price:5.2f} EUR -> {costs[-1]:7.2f} EUR per chair (baseline {as_arrays('baseline')['cost_per_unit_eur'].mean():.2f})")
print(f"   Break-even Stabilus price (linear between the runs): {np.interp(as_arrays('baseline')['cost_per_unit_eur'].mean(), costs, prices):.2f} EUR per spring")

print("\n(2) Where do the stand-still hours of the baseline come from?  Baseline with one change at a time")
line("Baseline as modelled", as_arrays("baseline"))
M2 = copy.deepcopy(M); M2["S"]["SUP-CN"]["sd_wd"] = 0.0
line("Ningbo lead time does not vary (sd 0)", variant(P, M2, dict(NO_DISRUPTION, switch_week=None)))
M2 = copy.deepcopy(M); M2["S"]["SUP-CN"]["sd_wd"] /= 2
line("Ningbo lead time varies half as much", variant(P, M2, dict(NO_DISRUPTION, switch_week=None)))
for sw in (0.5, 2.0):
    P2 = copy.deepcopy(P); P2["safety_weeks"] = sw
    line(f"Safety stock {sw} weeks instead of 1", variant(P2, M, dict(NO_DISRUPTION, switch_week=None)))

print("\n(3) Early warning in the disruption scenario: weeks of gas-spring cover at the end of the week (mean of 10 seeds)")
cover = np.mean([r["weekly"]["stock"] for r in RESULTS["disruption"]], axis=0) / P["demand_per_week"] / M["key_qty"]
hrs = np.mean([r["weekly"]["stopped_hours"] for r in RESULTS["disruption"]], axis=0)
for w in range(14, 21):
    print(f"   week {w + 1}: cover {cover[w]:4.2f} weeks, hours stopped in that week {hrs[w]:5.1f}")

(1) Why does the plant buy in Ningbo at all?  Same year without any disruption, German supplier from week 1
Baseline: Ningbo, no disruption                  699.8 chairs/week   149.93 EUR   4.60 kg CO2e   26.8 h stopped (0.0 - 53.7)
German supplier all year, no disruption          699.8 chairs/week   153.79 EUR   3.93 kg CO2e    0.0 h stopped (0.0 - 0.0)
   Ningbo Anatec Gas Spring Co., Ltd. price 10.00 + freight  7.45 = 17.45 EUR per spring delivered
   Stabilus GmbH Koblenz  price 22.00 + freight  0.25 = 22.25 EUR per spring delivered
   Ningbo, sea lane at the container rate only (no 0.18 EUR/tkm part): 10.00 +  2.90 = 12.90 EUR per spring delivered
   Which Stabilus price makes the switch (week 16, Suez closed) cost the same per chair as the baseline?
      Stabilus 14.50 EUR ->  148.47 EUR per chair (baseline 149.93)
      Stabilus 17.00 EUR ->  149.83 EUR per chair (baseline 149.93)
      Stabilus 17.50 EUR ->  150.10 EUR per chair (baseline 149.93)
      Stabilus 22.00 EUR ->  1

**What the numbers say**

1. **Why Ningbo?** A gas spring from Ningbo costs 17.45 EUR delivered (price 10.00 + freight 7.45), one from Stabilus 22.25 EUR (22.00 + 0.25). Even in the year with the closed canal a chair is cheaper with Ningbo (150.49 against 152.55 EUR), and a year with Stabilus only and no disruption would cost 153.79 instead of 149.93 EUR. The plant therefore buys in Ningbo because it is cheaper. The switch to the German supplier buys stability, not savings: 133.1 instead of 8.3 hours stopped, 9.2 chairs more a week and 0.41 kg less CO2e per chair, for 2.61 EUR more per chair than in the baseline (2.05 EUR more than in the disruption without a switch). At a Stabilus price of about 17.2 EUR the switch would cost the same per chair as the baseline; the 22.00 EUR is our estimate from a retail listing (`sources.csv`). The result does not depend on the guessed part of the sea freight (0.18 EUR per tonne-kilometre): with the container rate only, Ningbo would cost 12.90 EUR delivered and be cheaper still. What the model leaves out: contracts and minimum volumes, supplier approval and quality history, the capacity of the supplier, price negotiations.
2. **Why does the baseline stand still for 26.8 hours?** The order rule covers one week of safety stock plus the review period of two weeks, while the sea lead time varies by 3.5 working days (sd). Without variation the baseline never stops (0.0 h), with half the variation 0.3 h, with two weeks of safety stock 0.3 h, with half a week 102.4 h. The stand-still of the baseline is therefore a pure effect of lead-time variation against a one-week safety stock.
3. **Early warning.** Cover of gas springs falls from 2.4 weeks (week 15) to 1.8 (week 16) and 0.1 (week 18) while hours stopped only appear in weeks 17 to 19. The cover is the signal that comes first.

## Step 4 · Check cell
The baseline for the check cell is a list of 10 dictionaries, one per seed.

In [9]:
KEYS = ["weeks_simulated", "hours_simulated", "units_started", "units_finished",
        "key_components_used", "avg_units_on_line", "avg_hours_on_line", "cost_per_unit_eur"]
baseline = [{k: float(r[k]) for k in KEYS} for r in RESULTS["baseline"]]
print(pd.DataFrame(baseline).round(2).to_string())

   weeks_simulated  hours_simulated  units_started  units_finished  key_components_used  avg_units_on_line  avg_hours_on_line  cost_per_unit_eur
0             52.0           2080.0        36400.0         36393.0              36396.0              12.62               0.72             149.93
1             52.0           2080.0        36400.0         36391.0              36396.0              13.83               0.79             149.94
2             52.0           2080.0        36400.0         36393.0              36397.0              14.00               0.80             149.93
3             52.0           2080.0        36400.0         36391.0              36397.0               9.63               0.55             149.94
4             52.0           2080.0        36400.0         36392.0              36397.0               7.92               0.45             149.93
5             52.0           2080.0        36400.0         36392.0              36396.0               7.88               0.45     

In [10]:
# ===================== SSCM 2026 · GROUP PROJECT CHECK CELL · v1.0 =====================
# Copy this cell into your notebook UNCHANGED and run it after your baseline run.
# It needs two names defined earlier in your notebook:
#   DATA_DIR = the folder that holds your CSV files, e.g. DATA_DIR = "my_product_data"
#   baseline = your baseline results: one dictionary per seed (a list of 10), or one
#              dictionary with the means over the seeds. Keys (all numbers):
#     weeks_simulated      weeks in the run (52)
#     hours_simulated      clock hours of the run; the same clock as avg_hours_on_line
#     units_started        units that entered the line
#     units_finished       units that left the last station
#     key_components_used  key components taken from stock into the line
#     avg_units_on_line    time-average of units between start and finish
#     avg_hours_on_line    average hours from start to finish of a finished unit
#     cost_per_unit_eur    one-year total cost divided by units finished
# The grader runs the original of this cell on your files. Do not edit it.
import hashlib, math, os, datetime
import pandas as pd

CHECK_VERSION = "1.0"
REQUIRED = {
    "products.csv": ["product_id", "product_name", "frame_material", "mass_kg", "list_price_eur"],
    "bom.csv": ["product_id", "component_id", "component_name", "qty_per_unit", "mass_kg",
                "unit_cost_eur", "supplier_id", "added_at_workcenter"],
    "workcenters.csv": ["workcenter_id", "workcenter_name", "seq", "num_servers", "shift_hours",
                        "days_per_week", "availability", "process_time_cv", "fixed_cost_eur_week"],
    "routing.csv": ["product_id", "seq", "workcenter_id", "process_time_min", "setup_time_min"],
    "process_environment.csv": ["workcenter_id", "kw_running", "kw_idle", "waste_kg_per_unit",
                                "water_l_per_unit"],
    "suppliers.csv": ["supplier_id", "supplier_name", "country", "component_id", "unit_price_eur",
                      "promised_lead_days", "lead_time_mean_days", "lead_time_sd_days",
                      "capacity_per_month", "min_per_month", "kg_co2e_per_unit", "location_id"],
    "locations.csv": ["location_id", "name", "type", "country", "lat", "lon"],
    "network_lanes.csv": ["lane_id", "origin_id", "destination_id", "mode_id", "distance_km",
                          "transit_days_mean", "transit_days_sd"],
    "freight_rates.csv": ["lane_id", "mode_id", "rate_eur_per_shipment", "rate_eur_per_tkm",
                          "min_charge_eur"],
    "sources.csv": ["file", "row_id", "column", "value", "source_type", "source_or_calculation",
                    "sanity_check"],
}
MAY_BE_BLANK = {("bom.csv", "supplier_id")}           # parts not bought from a modelled supplier
MAY_BE_NEGATIVE = {("locations.csv", "lat"), ("locations.csv", "lon")}
KEYS = ["weeks_simulated", "hours_simulated", "units_started", "units_finished",
        "key_components_used", "avg_units_on_line", "avg_hours_on_line", "cost_per_unit_eur"]

def _cell_fingerprint():
    try:
        return hashlib.sha256(get_ipython().user_ns["_ih"][-1].strip().encode()).hexdigest()[:12]
    except Exception:
        return "n/a"

def _runs():
    b = globals().get("baseline")
    runs = b if isinstance(b, (list, tuple)) else [b]
    if b is None or not runs:
        raise ValueError("no `baseline` defined: run your baseline first")
    for i, r in enumerate(runs, 1):
        missing = [k for k in KEYS if k not in r]
        if missing:
            raise ValueError(f"baseline run {i} lacks {missing}")
        for k in KEYS:
            if not isinstance(r[k], (int, float)) or not math.isfinite(r[k]) or r[k] < 0:
                raise ValueError(f"baseline run {i}: {k} = {r[k]!r} is not a number ≥ 0")
        if r["units_finished"] <= 0 or r["hours_simulated"] <= 0 or r["weeks_simulated"] <= 0:
            raise ValueError(f"baseline run {i}: units_finished, hours and weeks must be > 0")
    return runs

def _verdict(name, ok, lines):
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
    for line in lines:
        print("      " + line)
    return ok

data, results = {}, {}
folder = globals().get("DATA_DIR", ".")
h = hashlib.sha256()

# ---- T1 Format ----------------------------------------------------------------------
problems = []
for fname, cols in REQUIRED.items():
    path = os.path.join(folder, fname)
    if not os.path.exists(path):
        problems.append(f"{fname}: file missing in {folder!r}")
        continue
    h.update(open(path, "rb").read())
    df = pd.read_csv(path)
    df.columns = [str(c) for c in df.columns]
    data[fname] = df
    lacking = [c for c in cols if c not in df.columns]
    if lacking:
        problems.append(f"{fname}: column(s) missing or misspelt: {lacking}")
    if len(df) == 0:
        problems.append(f"{fname}: no rows")
    for c in [c for c in cols if c in df.columns]:
        blanks = df[c].isna() | (df[c].astype(str).str.strip() == "")
        if blanks.any() and (fname, c) not in MAY_BE_BLANK:
            problems.append(f"{fname}: {c} is empty in row(s) {list(df.index[blanks] + 2)}")
        num = pd.to_numeric(df[c], errors="coerce")
        if (num < 0).any() and (fname, c) not in MAY_BE_NEGATIVE:
            problems.append(f"{fname}: {c} is negative in row(s) {list(df.index[num < 0] + 2)}")
if "workcenters.csv" in data and "availability" in data["workcenters.csv"]:
    av = pd.to_numeric(data["workcenters.csv"]["availability"], errors="coerce")
    if ((av <= 0) | (av > 1) | av.isna()).any():
        problems.append("workcenters.csv: availability must be a share between 0 and 1")
if "routing.csv" in data and "process_time_min" in data["routing.csv"]:
    if (pd.to_numeric(data["routing.csv"]["process_time_min"], errors="coerce") <= 0).any():
        problems.append("routing.csv: every process_time_min must be above 0")
results["T1"] = _verdict("T1 Format", not problems,
                         problems[:12] + (["… and more"] if len(problems) > 12 else [])
                         or ["all 10 files present, columns exact, no blanks or negatives"])

# ---- baseline numbers for T2–T5 -------------------------------------------------------
try:
    runs = _runs()
    base_error = None
except Exception as e:
    runs, base_error = [], str(e)
if base_error:
    for t in ["T2 Capacity", "T3 Little's Law", "T4 Flow balance", "T5 Margin"]:
        results[t[:2]] = _verdict(t, False, [base_error])
else:
    try:
        prod = data["products.csv"].iloc[0]
        pid = prod["product_id"]
        wc = data["workcenters.csv"].set_index("workcenter_id")
        rt = data["routing.csv"]
        rt = rt[rt["product_id"] == pid]
        minutes = rt.groupby("workcenter_id")["process_time_min"].sum()
        unknown = [w for w in minutes.index if w not in wc.index]
        avail_min_week = (wc["num_servers"] * wc["shift_hours"] * 60 * wc["days_per_week"]
                          * wc["availability"])
        # ---- T2 Capacity: load offered by the units started, per station
        lines, ok = [], not unknown and len(minutes) > 0
        if unknown:
            lines.append(f"routing.csv names stations not in workcenters.csv: {unknown}")
        worst = {}
        for r in runs:
            for w, m in minutes.items():
                if w in wc.index:
                    share = r["units_started"] * m / (avail_min_week[w] * r["weeks_simulated"])
                    worst[w] = max(worst.get(w, 0), share)
        for w, share in worst.items():
            flag = "  <- 100 % or more" if share >= 1 else ""
            lines.append(f"{w} {wc.loc[w, 'workcenter_name']}: busy {share:.1%} of available time{flag}")
            ok = ok and share < 1
        results["T2"] = _verdict("T2 Capacity", ok, lines)
        # ---- T3 Little's Law: L = λ × W, within 10 %
        lines, ok = [], True
        for i, r in enumerate(runs, 1):
            lam = r["units_finished"] / r["hours_simulated"]
            lw = lam * r["avg_hours_on_line"]
            gap = abs(r["avg_units_on_line"] - lw) / lw if lw > 0 else float("inf")
            ok = ok and gap <= 0.10
            if (gap > 0.10 and len(lines) < 3) or i == 1:
                lines.append(f"run {i}: L = {r['avg_units_on_line']:.2f}  vs  λ×W = {lam:.3f}/h × "
                             f"{r['avg_hours_on_line']:.2f} h = {lw:.2f}  (gap {gap:.1%})")
        results["T3"] = _verdict("T3 Little's Law", ok, lines + [f"{len(runs)} run(s) checked"])
        # ---- T4 Flow balance
        bom = data["bom.csv"]
        bom = bom[bom["product_id"] == pid]
        keys = bom[bom["component_id"].isin(data["suppliers.csv"]["component_id"])]
        lines, ok = [], len(keys) == 1
        if len(keys) != 1:
            lines.append(f"expected one key component (in bom.csv and suppliers.csv), found {len(keys)}")
        else:
            q = float(keys["qty_per_unit"].iloc[0])
            n_opt = int((data["suppliers.csv"]["component_id"] == keys["component_id"].iloc[0]).sum())
            if n_opt < 2:
                lines.append(f"note: only {n_opt} sourcing option in suppliers.csv (the brief asks for two)")
            for i, r in enumerate(runs, 1):
                s, f, u = r["units_started"], r["units_finished"], r["key_components_used"]
                good = f <= s + 0.5 and f * q - 0.5 <= u <= s * q + 0.5
                ok = ok and good
                if not good or i == 1:
                    lines.append(f"run {i}: started {s:,.0f}, finished {f:,.0f}, key components used "
                                 f"{u:,.0f} (allowed {f*q:,.0f} to {s*q:,.0f})")
        results["T4"] = _verdict("T4 Flow balance", ok, lines)
        # ---- T5 Margin
        price = float(prod["list_price_eur"])
        material = float((bom["qty_per_unit"] * bom["unit_cost_eur"]).sum())
        costs = [r["cost_per_unit_eur"] for r in runs]
        lines = [f"cost per unit {min(costs):,.2f} to {max(costs):,.2f} EUR vs list price {price:,.2f} EUR"]
        if min(costs) < material:
            lines.append(f"warning: below the material cost alone ({material:,.2f} EUR per unit)")
        results["T5"] = _verdict("T5 Margin", max(costs) < price, lines)
    except Exception as e:
        for t in ["T2 Capacity", "T3 Little's Law", "T4 Flow balance", "T5 Margin"]:
            if t[:2] not in results:
                results[t[:2]] = _verdict(t, False, [f"could not run: {e}"])

print("-" * 78)
print(f"{sum(results.values())} of 5 tests passed  ·  {'ALL PASS' if all(results.values()) and len(results) == 5 else 'NOT YET'}")
print(f"check cell v{CHECK_VERSION} · cell {_cell_fingerprint()} · data {h.hexdigest()[:12]} · "
      f"{datetime.datetime.now():%Y-%m-%d %H:%M}")
# ======================= end of check cell · do not edit above ==========================

PASS  T1 Format
      all 10 files present, columns exact, no blanks or negatives
PASS  T2 Capacity
      WC1 PU Foam Moulding & Trimming: busy 81.0% of available time
      WC2 Upholstery & Fabric Cover Assembly: busy 72.9% of available time
      WC3 Gas Spring Base Casters & Tilt Mechanism Sub-Assembly: busy 77.8% of available time
      WC4 Backrest Joining Armrest & Ergonomic Test: busy 66.6% of available time
      WC5 Packaging: busy 68.1% of available time
PASS  T3 Little's Law
      run 1: L = 12.62  vs  λ×W = 17.497/h × 0.72 h = 12.60  (gap 0.2%)
      10 run(s) checked
PASS  T4 Flow balance
      run 1: started 36,400, finished 36,393, key components used 36,396 (allowed 36,393 to 36,400)
PASS  T5 Margin
      cost per unit 149.93 to 149.94 EUR vs list price 310.08 EUR
------------------------------------------------------------------------------
5 of 5 tests passed  ·  ALL PASS
check cell v1.0 · cell 3ad30383c119 · data b923a5690b12 · 2026-11-10 21:31


## Step 5 · `results.json` and animation

In [11]:
import json
# Which seed runs in the movie? The seed whose hours stopped in the disruption are closest to the median of the 10 seeds.
dis = np.array([r["hours_stopped"] for r in RESULTS["disruption"]])
ANIM_IDX = int(np.argmin(np.abs(dis - np.median(dis))))
ANIM_SEED = P["seeds"][ANIM_IDX]

def short(sid):
    s = M["S"][sid]; return " ".join(s["name"].replace(",", "").split()[:2]) + " (" + s["country"] + ")"

def scen_block(sc):
    runs = RESULTS[sc]
    mean = {k: float(np.mean([r[k] for r in runs])) for k, _, _ in SCORE}
    lo = {k: float(np.min([r[k] for r in runs])) for k, _, _ in SCORE}
    hi = {k: float(np.max([r[k] for r in runs])) for k, _, _ in SCORE}
    w = runs[ANIM_IDX]["weekly"]
    return dict(label=P["scenarios"][sc]["label"], scoreboard=mean, scoreboard_min=lo, scoreboard_max=hi,
                weeks=dict(queue=w["queue"], in_transit=w["in_transit"], stock=w["stock"], finished=w["finished"],
                           stopped_hours=w["stopped_hours"], supplier_down=w["supplier_down"],
                           supplier=[short(s) for s in w["supplier"]]))

results = dict(
    product=M["product"], unit="chairs", key_component=M["key_name"],
    supplier=short(M["base_supplier"]), transport="ship via Suez + truck",
    event="Red Sea diversions 2024: ships sail around the Cape (+10-14 days); alternative supplier Stabilus (Koblenz) by truck",
    stations=[s["name"] for s in M["st"]], key_station_index=M["key_idx"],
    parts_per_square=P["demand_per_week"], n_seeds=len(P["seeds"]), animation_seed=ANIM_SEED,
    scenarios={sc: scen_block(sc) for sc in P["scenarios"]})
with open("results.json", "w") as f:
    json.dump(results, f, indent=1)
print("results.json written · the movie shows seed", ANIM_SEED)

results.json written · the movie shows seed 2


In [12]:
ANIM_TEMPLATE = r'''<div id="sscm" style="font-family:system-ui,Arial,sans-serif;background:#fff;color:#222;max-width:780px;margin:auto;padding:14px;border-radius:8px">
<div id="ttl" style="font-size:20px;font-weight:700;color:#333"></div>
<div id="sub" style="font-size:12px;color:#555;margin:2px 0 8px"></div>
<div id="ans" style="font-size:14px;line-height:1.45;background:#f6f6f6;border-left:4px solid #2a6fdb;border-radius:4px;padding:8px 12px;margin:0 0 10px"></div>
<div id="btns" style="display:flex;gap:8px;flex-wrap:wrap;margin-bottom:10px"></div>
<div style="display:flex;align-items:center;gap:12px;margin-bottom:6px;font-size:13px">
  <button id="play" style="padding:5px 12px;border:1px solid #bbb;background:#fff;border-radius:4px;cursor:pointer;color:#222">&#9654; Play</button>
  <span id="wk" style="min-width:70px;font-weight:600"></span>
  <input id="sl" type="range" min="0" value="0" style="flex:1">
  <label>speed <select id="spd"><option value="700">slow</option><option value="350" selected>normal</option><option value="120">fast</option></select></label>
</div>
<svg id="stage" viewBox="0 0 760 200" width="100%" style="display:block"></svg>
<div id="note" style="font-size:11px;color:#777;margin:2px 0 8px"></div>
<div id="kpi" style="display:grid;grid-template-columns:repeat(4,1fr);gap:8px"></div>
<svg id="chart" viewBox="0 0 760 170" width="100%" style="display:block;margin-top:8px"></svg>
<div id="foot" style="font-size:11px;color:#777;margin-top:4px"></div>
</div>
<script>
(function(){
const D = __DATA__;
const keys = Object.keys(D.scenarios);
const COL = {baseline:'#444444', disruption:'#d62839', countermeasure:'#2a6fdb'};
const $ = id => document.getElementById(id);
let sc = keys[0], wk = 0, timer = null;
const N = D.scenarios[keys[0]].weeks.finished.length;
const maxStock = Math.max(1, ...keys.map(k => Math.max(...D.scenarios[k].weeks.stock)));
const maxFin = Math.max(1, ...keys.map(k => Math.max(...D.scenarios[k].weeks.finished)));
const unit = D.unit;
$('ttl').textContent = D.product + ' · line and supply chain';
$('sub').textContent = 'Key component: ' + D.key_component + ' · ' + D.event + ' · one year, week by week';
$('foot').textContent = 'On the queues, one dot = 5 ' + unit + ' waiting. Numbers from results.json (mean of ' + D.n_seeds + ' seeds in the scoreboard; the movie shows seed ' + D.animation_seed + '). Line stood still = a part is missing at S' + (D.key_station_index+1) + ': a chair waits there for a gas spring, the stock is empty and no place of S' + (D.key_station_index+1) + ' is working.';
$('sl').max = N - 1;
const sb = (m, d) => keys.map(k => D.scenarios[k].scoreboard[m].toFixed(d)).join(' &rarr; ');
$('ans').innerHTML = '<b>Three numbers change</b> (' + keys.map(k => D.scenarios[k].label).join(' &rarr; ') + ', mean of ' + D.n_seeds + ' seeds):<br>' +
  'hours the line stood still <b>' + sb('hours_stopped',1) + '</b> &middot; ' + unit + ' per week <b>' + sb('units_per_week',1) + '</b> &middot; kg CO&#8322;e per unit <b>' + sb('kg_co2e_per_unit',2) + '</b>';
keys.forEach(k => {
  const b = document.createElement('button'); b.textContent = D.scenarios[k].label; b.dataset.k = k;
  b.style.cssText = 'padding:5px 12px;border:1px solid #bbb;border-radius:4px;cursor:pointer;font-size:13px';
  b.onclick = () => { sc = k; draw(); }; $('btns').appendChild(b);
});
function box(x,y,w,h,fill,stroke){return `<rect x="${x}" y="${y}" width="${w}" height="${h}" rx="5" fill="${fill}" stroke="${stroke||'none'}" stroke-width="1.5"/>`}
function txt(x,y,s,sz,col,anchor,wt){return `<text x="${x}" y="${y}" font-size="${sz}" fill="${col||'#333'}" text-anchor="${anchor||'middle'}" font-weight="${wt||400}">${s}</text>`}
function drawStage(){
  const S = D.scenarios[sc], W = S.weeks, i = wk;
  const down = W.supplier_down[i], stopped = W.stopped_hours[i] > 0;
  let s = '';
  // supplier
  const supName = (W.supplier && W.supplier[i]) || D.supplier;
  s += box(8,50,120,70,'#f4f4f4',down?'#d62839':'#888');
  s += txt(68,70,'Supplier',12,'#222','middle',700) + txt(68,86,supName,9,'#444') +
       txt(68,106,down ? 'Suez closed' : 'delivering',10,down?'#d62839':'#2e8b57','middle',700);
  // transit
  const nTr = Math.min(7, Math.ceil(W.in_transit[i] / (D.parts_per_square || 700)));
  s += txt(172,60,'in transit: ' + W.in_transit[i],10,'#444');
  s += `<line x1="132" y1="85" x2="212" y2="85" stroke="${down?'#d62839':'#999'}" stroke-width="2" stroke-dasharray="6 4"/>`;
  for (let k=0;k<nTr;k++) s += box(134 + k*11, 79, 9, 12, down ? '#d62839' : '#555');
  // stock bucket
  const bh = 110, fill = Math.round(bh * W.stock[i] / maxStock);
  s += `<rect x="230" y="30" width="56" height="${bh}" fill="#fff" stroke="#555" stroke-width="1.5"/>`;
  s += `<rect x="231.5" y="${30+bh-fill}" width="53" height="${fill}" fill="${W.stock[i]===0?'#d62839':'#5b8fd0'}"/>`;
  s += txt(258,24,'Stock',11,'#222','middle',700) + txt(258,156,W.stock[i] + ' parts',10,'#444');
  // stations
  const x0 = 312, dx = 78;
  s += `<line x1="${x0-10}" y1="85" x2="${x0+4*dx+80}" y2="85" stroke="#ddd" stroke-width="5"/>`;
  D.stations.forEach((name,j) => {
    const x = x0 + j*dx, q = W.queue[i][j], isKey = j === D.key_station_index;
    const red = isKey && stopped;
    s += txt(x+34,50,Math.round(q) + ' waiting',9,'#a05a00');
    const nd = Math.min(10, Math.round(q/5));
    for (let d=0; d<nd; d++) s += `<circle cx="${x+10+d*6}" cy="60" r="2.6" fill="#d62839"/>`;
    s += box(x,70,68,30,red?'#d62839':'#999') + txt(x+34,89,'S'+(j+1),10,'#fff','middle',700);
    const words = name.split(' ').slice(0,3).join(' ');
    s += txt(x+34,116,words.length>16 ? words.slice(0,16)+'…' : words,8.5,'#444');
  });
  // finished
  const fx = x0 + 5*dx + 4;
  s += txt(fx,60,'Finished',10,'#222','start',700) + txt(fx,84,W.finished[i] ,22,'#222','start',700) +
       txt(fx,98,'this week',9,'#555','start');
  const cum = W.finished.slice(0,i+1).reduce((a,b)=>a+b,0);
  s += txt(fx,112,Math.round(cum).toLocaleString('en-US') + ' so far',9,'#777','start');
  $('stage').innerHTML = s;
  $('note').textContent = 'Scenario: ' + S.label + (stopped ? ' — line waits for parts (' + W.stopped_hours[i].toFixed(1) + ' h this week)' : '');
}
function drawKpi(){
  const S = D.scenarios[sc], b = S.scoreboard, lo = S.scoreboard_min, hi = S.scoreboard_max;
  const f = (v,d) => v.toFixed(d);
  const rng = (k,d) => lo ? 'range ' + f(lo[k],d) + ' – ' + f(hi[k],d) : '';
  const T = [[f(b.units_per_week,1), unit + ' per week', rng('units_per_week',1)],
             [f(b.cost_per_unit_eur,2) + ' €', 'total cost per unit (one year)', rng('cost_per_unit_eur',2)],
             [f(b.kg_co2e_per_unit,2), 'kg CO₂e per unit', rng('kg_co2e_per_unit',2)],
             [f(b.hours_stopped,1) + ' h', 'line stood still (year)', rng('hours_stopped',1)]];
  $('kpi').innerHTML = T.map(t => `<div style="background:#f1f1f1;border-radius:6px;padding:8px 10px"><div style="font-size:20px;font-weight:700">${t[0]}</div><div style="font-size:11px;color:#444">${t[1]}</div><div style="font-size:10px;color:#888">${t[2]}</div></div>`).join('');
}
function drawChart(){
  const X0=46,X1=740,Y0=14,Y1=120; let s='';
  const x = w => X0 + (X1-X0)*w/(N-1), y = v => Y1 - (Y1-Y0)*v/maxFin;
  s += txt(X0,9,unit + ' finished per week',9,'#555','start');
  [0,0.5,1].forEach(g => { const v = maxFin*g; s += `<line x1="${X0}" y1="${y(v)}" x2="${X1}" y2="${y(v)}" stroke="#eee"/>` + txt(X0-4,y(v)+3,Math.round(v),8,'#777','end'); });
  keys.forEach(k => {
    const d = D.scenarios[k].weeks.finished.map((v,w) => (w?'L':'M') + x(w).toFixed(1) + ' ' + y(v).toFixed(1)).join(' ');
    s += `<path d="${d}" fill="none" stroke="${COL[k]||'#888'}" stroke-width="${k===sc?2.4:1.2}" opacity="${k===sc?1:0.55}"/>`;
  });
  s += `<line x1="${x(wk)}" y1="${Y0}" x2="${x(wk)}" y2="${Y1}" stroke="#555" stroke-dasharray="3 3"/>`;
  keys.forEach((k,j) => { const lx = X0 + j*230; s += `<rect x="${lx}" y="140" width="12" height="4" fill="${COL[k]||'#888'}"/>` + txt(lx+16,146,D.scenarios[k].label,9,'#444','start'); });
  s += txt(X0,160,'week 1',8,'#777','start') + txt(X1,160,'week ' + N,8,'#777','end');
  $('chart').innerHTML = s;
}
function draw(){
  $('wk').textContent = 'Week ' + (wk+1); $('sl').value = wk;
  [...$('btns').children].forEach(b => { const on = b.dataset.k === sc; b.style.background = on ? '#111' : '#fff'; b.style.color = on ? '#fff' : '#222'; });
  drawStage(); drawKpi(); drawChart();
}
function stop(){ clearInterval(timer); timer = null; $('play').innerHTML = '&#9654; Play'; }
function start(){ if (wk >= N-1) wk = 0; $('play').innerHTML = '&#10074;&#10074; Pause';
  timer = setInterval(() => { if (wk < N-1) { wk++; draw(); } else stop(); }, +$('spd').value); }
$('play').onclick = () => timer ? stop() : start();
$('sl').oninput = e => { wk = +e.target.value; draw(); };
$('spd').onchange = () => { if (timer) { stop(); start(); } };
draw();
})();
</script>
'''

In [13]:
from IPython.display import HTML, display
html = ANIM_TEMPLATE.replace("__DATA__", json.dumps(json.load(open("results.json"))))
open("animation.html", "w", encoding="utf-8").write("<!doctype html><meta charset='utf-8'><body style='background:#fff'>" + html + "</body>")
display(HTML(html))
print("animation.html saved: download it from the file panel on the left")

animation.html saved: download it from the file panel on the left


## Reading the results

* **Resource reconfiguration (Ambulkar et al., 2015):** switching to Stabilus is a reconfiguration of the supply base. The numbers that show its effect are the hours the line stood still and the chairs per week in the table of Step 3: the disruption costs hours of stand-still and some chairs, the countermeasure brings both back to the baseline level. It also lowers CO2 per chair, because the long sea route disappears (see the CO2 table). The price is cost: a chair is 2.61 EUR dearer than in the baseline, because a Stabilus spring is dearer than a Ningbo spring even with the freight saved (cost table, key_material and freight).
* **Measuring (Schoenherr et al., 2023):** the earliest measurable warning is the cover of gas springs in weeks of demand (stock ÷ weekly demand), not the hours stopped. In the disruption scenario the mean cover is 2.4 weeks at the end of week 15, 1.8 in week 16 and 0.1 in week 18 (Step 3b, part 3), while stand-still hours only show up from weeks 17 to 19. A plant that tracks the cover can decide on the switch before the line stops; with a switch in week 16 the cover is still 1.8 weeks.

# References / Literature

* Ambulkar, S., Blackhurst, J., & Grawe, S. (2015). Firm's resilience to supply chain disruptions: Scale development and empirical examination. *Journal of Operations Management*, 33–34, 111–122. https://doi.org/10.1016/j.jom.2014.11.002
* Schoenherr, T., Mena, C., Vakil, B., & Choi, T. Y. (2023). Creating resilient supply chains through a culture of measuring. *Journal of Purchasing and Supply Management*, 29(4), 100824. https://doi.org/10.1016/j.pursup.2023.100824

**Sources of the disruption scenario** (accessed November 2026):

* Automotive Logistics. (2024). *Vessel operators divert ships from Red Sea following Houthi attacks* (Wallenius Wilhelmsen: rerouting adds 10–14 days; Maersk pause of 2 January 2024). https://www.automotivelogistics.media/logistics-service-providers/vessel-operators-divert-ships-from-red-sea-following-houthi-attacks/209621
* Lloyd's List. (2023, December 15). *Maersk and Hapag-Lloyd halt Red Sea passages*. https://www.lloydslist.com/LL1147624/Maersk-and-Hapag-Lloyd-halt-Red-Sea-passages (reprint by Kuehne+Nagel: https://mykn.kuehne-nagel.com/news/article/maersk-and-hapaglloyd-halt-red-sea-passages-15-Dec-2023)
* Trans.info. (2023, December). *Red Sea sailings* (Drewry: the longer route adds 10–14 days). https://trans.info/en/red-sea-sailings-375101
* Reuters. (2024, January 12). *Tesla Berlin to stop most output for two weeks due to Red Sea disruption*. https://www.businessday.co.za/bd/companies/2024-01-12-tesla-berlin-to-stop-most-output-for-two-weeks-due-to-red-sea-disruption/
* Detour distance of the Cape route (model: 3,400 nautical miles). We could not retrieve the original CNBC article (December 2023, quoting M. Burgess of C.H. Robinson), so the figure is checked against three sources we could read: C.H. Robinson (2024), *Red Sea and Panama Canal Disruptions: 2024 Implications for Shippers*, about 3,500 additional nautical miles, https://www.chrobinson.com/en-au/resources/blog/shipping-disruptions-red-sea-panama-canal/ ; gCaptain (2024, January 19), *Suez Canal diversions: Drewry assesses impact on container shipping*, nearly 3,600 nautical miles longer, https://gcaptain.com/suez-canal-diversions-drewry-assesses-impact-on-container-shipping ; Portcast, *Impact from avoiding Suez and going around South Africa*, Singapore–Rotterdam 11,755 instead of 8,288 nautical miles (3,467 more), https://www.portcast.io/blog/safety-disruption-in-bab-al-mandeb-strait-impact-from-avoiding-suez-and-going-around-south-africa
* The Reuters article on Tesla Berlin is quoted as published by BusinessDay (see above); we did not verify the original reuters.com address.